# Chương 4. Phân tích thống kê văn bản (Text EDA)

**Dữ liệu.** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews): 50.000 review phim tiếng Anh, 2 nhãn `positive` / `negative`.

**Nguyên tắc của chương.** Khử trùng lặp rồi chia train/test **ngay ở mục 4.1**, sau đó mọi phân tích khám phá (độ dài, từ phổ biến, TTR, Zipf) **chỉ dùng tập train**. Lý do: các kết luận của EDA (ví dụ "`br` là từ phổ biến thứ 7") dẫn tới quyết định tiền xử lý ở chương 5; nếu tính trên cả tập test thì quyết định đó đã "nhìn thấy" test (rò rỉ gián tiếp). Tập test được giữ nguyên, chỉ dùng để đánh giá ở các chương sau.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dmprep import config
from dmprep.data import load_imdb
import re
from collections import Counter
from itertools import chain

from scipy import stats
from sklearn.model_selection import train_test_split
from wordcloud import STOPWORDS, WordCloud

config.set_seed()
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

RESULTS: dict = {}  # mọi score / p-value của chương này được lưu vào đây

## 4.1 Tổng quan dữ liệu, khử trùng lặp và chia train/test

**Trùng lặp.** Cùng một review xuất hiện nhiều lần sẽ làm điểm đánh giá bị thổi phồng nếu một bản nằm ở train và bản kia ở test (mô hình chỉ cần "nhớ"). Ta xác định trùng lặp bằng **khóa chuẩn hóa**: bỏ thẻ HTML, chuyển chữ thường, gộp khoảng trắng. Khóa này bắt được cả các bản chỉ khác nhau ở định dạng, nên rộng hơn trùng lặp chính xác. Với mỗi nhóm trùng ta giữ bản đầu tiên; nếu một nhóm có **nhãn mâu thuẫn** thì loại cả nhóm vì không biết nhãn đúng.

**Khử trùng lặp phải làm trước khi chia.** Nếu chia trước, các bản sao có thể rơi vào cả hai tập.

**Chia phân tầng 80/20.** `train_test_split(..., stratify=label, random_state=config.SEED)` giữ nguyên tỉ lệ nhãn ở cả hai tập và tái lập được. Tập test chỉ dùng để đánh giá cuối cùng ở các chương sau.

**Dấu hiệu nhiễu.** Ta đo tỉ lệ review chứa thẻ HTML, URL, mention, hashtag, chữ số và ký tự ngoài ASCII; kết quả quyết định các bước chuẩn hóa có điều kiện ở chương 5.

In [2]:
df = load_imdb()
print("Kích thước:", df.shape, "| số ô thiếu:", int(df.isna().sum().sum()))
print(df["sentiment"].value_counts().to_dict())
RESULTS.update({"n_samples": len(df), "class_counts": df["sentiment"].value_counts().to_dict()})


def normalize_key(s: pd.Series) -> pd.Series:
    """Khóa so sánh trùng lặp: bỏ thẻ HTML, chữ thường, gộp khoảng trắng."""
    return s.str.replace(r"<[^>]+>", " ", regex=True).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()


df["key"] = normalize_key(df["review"])
n_dup_exact = int(df.duplicated(subset="review").sum())
n_dup_key = int(df.duplicated(subset="key").sum())
label_per_key = df.groupby("key")["label"].nunique()
conflict_keys = label_per_key[label_per_key > 1].index
print(f"Review thừa so với bản đầu tiên: trùng chính xác = {n_dup_exact}, trùng theo khóa chuẩn hóa = {n_dup_key}; nhóm có nhãn mâu thuẫn = {len(conflict_keys)}")
RESULTS.update({"n_duplicates_exact": n_dup_exact, "n_duplicates_normalized": n_dup_key, "n_label_conflict_groups": len(conflict_keys)})

# Dấu hiệu nhiễu (tỉ lệ review có chứa)
patterns = {"thẻ HTML <br />": r"<br\s*/?>", "URL": r"https?://|www\.", "@mention": r"@\w+", "#hashtag": r"#\w+", "chữ số": r"\d", "ký tự ngoài ASCII": r"[^\x00-\x7f]"}
noise = pd.Series({k: df["review"].str.contains(p, regex=True).mean() * 100 for k, p in patterns.items()}, name="% review")
display(noise.round(2).to_frame())
RESULTS["noise_pct"] = noise.round(3).to_dict()

# Khử trùng lặp TRƯỚC khi chia, rồi chia phân tầng
df_clean = df[~df["key"].isin(conflict_keys)].drop_duplicates(subset="key").drop(columns="key").reset_index(drop=True)
train_idx, test_idx = train_test_split(df_clean.index, test_size=config.TEST_SIZE, stratify=df_clean["label"], random_state=config.SEED)
df_clean["split"] = "train"
df_clean.loc[test_idx, "split"] = "test"
df_clean[["review", "sentiment", "label", "split"]].to_parquet(config.PROCESSED_DIR / "imdb_split.parquet", index=False)

split_table = df_clean.groupby("split").agg(n=("label", "size"), positive_pct=("label", lambda x: x.mean() * 100))
display(split_table.round(2))
RESULTS.update({"n_after_dedup": len(df_clean), "n_train": int((df_clean["split"] == "train").sum()), "n_test": int((df_clean["split"] == "test").sum()),
                "positive_pct_train": float(split_table.loc["train", "positive_pct"]), "positive_pct_test": float(split_table.loc["test", "positive_pct"])})

train = df_clean[df_clean["split"] == "train"].copy()  # từ đây trở đi mọi EDA chỉ dùng tập train
pos, neg = train[train["label"] == 1], train[train["label"] == 0]
print(f"EDA dùng tập train: {len(train)} review (positive {len(pos)}, negative {len(neg)})")

Kích thước: (50000, 3) | số ô thiếu: 0
{'positive': 25000, 'negative': 25000}


Review thừa so với bản đầu tiên: trùng chính xác = 418, trùng theo khóa chuẩn hóa = 422; nhóm có nhãn mâu thuẫn = 0


,% review
thẻ HTML <br />,58.40
URL,0.40
@mention,0.16
#hashtag,0.48
chữ số,56.01
ký tự ngoài ASCII,9.32


,n,positive_pct
split,,
test,9916,50.19
train,39662,50.19


EDA dùng tập train: 39662 review (positive 19905, negative 19757)


**Phân tích kết quả.**

- **Cân bằng lớp.** 50.000 review, 25.000 positive và 25.000 negative, không thiếu ô nào. Sau khử trùng lặp còn **49.578 review**: train 39.662 (positive 50.19%) và test 9.916 (positive 50.19%), tức phép chia phân tầng giữ nguyên tỉ lệ nhãn.
- **Trùng lặp.** Có 418 review trùng chính xác và 422 review thừa theo khóa chuẩn hóa, tức có 4 bản chỉ khác nhau ở thẻ HTML, chữ hoa hoặc khoảng trắng mà so sánh chính xác không bắt được. Không nhóm trùng nào có nhãn mâu thuẫn, nên không phải loại cả nhóm. Con số 418 ở bản trước của dự án (chỉ so sánh chính xác) vì vậy được thay bằng 422.
- **Dấu hiệu nhiễu (trên 50.000 review gốc).** Thẻ HTML `<br />` có ở 58.40% review; chữ số ở 56.01%; ký tự ngoài ASCII ở 9.32%; hashtag 0.48%, URL 0.40%, mention 0.16%. Hệ quả cho chương 5: thẻ HTML bắt buộc phải bỏ; URL, mention và hashtag quá hiếm (dưới 0.5%) nên gần như không ảnh hưởng từ vựng; chữ số xuất hiện ở hơn một nửa số review và có thể mang thông tin (ví dụ "10/10"), nên việc xử lý số cần có điều kiện và được đo bằng thực nghiệm thay vì xóa mặc định.

## Lưu kết quả

Toàn bộ con số của chương được ghi ra `data/processed/results/` để chương báo cáo tham chiếu.

In [3]:
config.save_results("04_text_eda", RESULTS)
RESULTS

{'n_samples': 50000,
 'class_counts': {'positive': 25000, 'negative': 25000},
 'n_duplicates_exact': 418,
 'n_duplicates_normalized': 422,
 'n_label_conflict_groups': 0,
 'noise_pct': {'thẻ HTML <br />': 58.4,
  'URL': 0.404,
  '@mention': 0.164,
  '#hashtag': 0.48,
  'chữ số': 56.01,
  'ký tự ngoài ASCII': 9.32},
 'n_after_dedup': 49578,
 'n_train': 39662,
 'n_test': 9916,
 'positive_pct_train': 50.18657657203368,
 'positive_pct_test': 50.19160951996773}